In [ ]:
# 1. Install Dependencies
# We pin datasets to 2.19.0 because newer versions (3.0+) removed support for loading scripts (like fleurs.py)
!pip install datasets==2.19.0
!pip install git+https://github.com/huggingface/transformers
!pip install librosa
!pip install evaluate>=0.30
!pip install jiwer
!pip install accelerate -U

In [ ]:
# 2. Login to Hugging Face (Required for Common Voice)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 3. Load Dataset (Google FLEURS - Latin American Spanish)
from datasets import load_dataset, Audio

# Load FLEURS (es_419 is Latin American Spanish)
# We use trust_remote_code=True because FLEURS uses a loading script
ds_es = load_dataset("google/fleurs", "es_419", split="train+validation", trust_remote_code=True)

# Remove unused columns to save memory
ds_es = ds_es.remove_columns(["id", "num_samples", "path", "raw_transcription", "gender", "lang_id", "lang_group_id"])

# Resample to 16kHz (Whisper requirement)
ds_es = ds_es.cast_column("audio", Audio(sampling_rate=16000))

print(ds_es)

In [ ]:
# 4. Prepare Processor & Model
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id, language="Spanish", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_id)

# Freeze encoder to save memory (optional, good for Colab free tier)
model.freeze_encoder()

In [ ]:
# 5. Preprocessing Function
def prepare_dataset(batch):
    # Load audio
    audio = batch["audio"]
    
    # Compute log-Mel input features
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    
    # Encode target text (FLEURS uses 'transcription' column)
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

# Apply to dataset (this might take a while)
ds_es_prepared = ds_es.map(prepare_dataset, remove_columns=ds_es.column_names)

In [ ]:
# 6. Data Collator
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 to ignore loss
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        
        # Remove BOS token if present (Whisper adds it automatically)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
# 7. Training Arguments & Trainer
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-es-finetune",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=4000,
    fp16=True,
    eval_strategy="steps", # Updated from evaluation_strategy
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=ds_es_prepared,
    eval_dataset=ds_es_prepared, # In real scenario, split your dataset!
    data_collator=data_collator,
    processing_class=processor.feature_extractor, # Updated from tokenizer
)

trainer.train()

In [ ]:
# 8. Save Model
model.save_pretrained("./whisper-small-es-finetune-final")
processor.save_pretrained("./whisper-small-es-finetune-final")

# Optional: Push to Hub
# model.push_to_hub("your-username/whisper-small-es-finetune")
# processor.push_to_hub("your-username/whisper-small-es-finetune")